# Legendplex

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(

            library(stats), 
            library(emmeans), 
            library(outliers), # Grubbs outlier detection 
            
            # Data 
            library(tidyverse), 
            library(data.table), 
            library(reactable), 

            # Plotting 
            library(ComplexHeatmap), 
            library(patchwork), 
            library(cowplot), 
            library(ggrepel)

        )
    )
)

In [3]:
random_seed <- 42
set.seed(random_seed)

In [4]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20200109SPLENO")

In [5]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Parameter 

In [6]:
color$sample_group<- c("D0 +/+"="#66c2a5", "D0 cre/+"="#00634A", "D1 +/+"="#cd34b5", "D1 cre/+"="#FFAC1E", "D3 +/+"="#cd34b5", "D3 cre/+"="#FFAC1E", "D6 +/+"="#cd34b5", "D6 cre/+"="#FFAC1E")

# Function 

In [90]:
iqr <- function(data) {

    data <- data %>%
    
        group_by(sample_group, measurement) %>%
        mutate(
            
            Q1=quantile(value, 0.25, na.rm=TRUE),
            Q3=quantile(value, 0.75, na.rm=TRUE),
            IQR=Q3 - Q1,
            # outlier=value < (Q1 - 1.5*IQR) | value > (Q3 + 1.5*IQR)
            outlier=value > (Q3 + 1.5*IQR)
        
        ) %>% dplyr::filter(!outlier)

    return(data)
}

In [8]:
data_stat <- function(data) {

    data <- data %>% dplyr::group_by(measurement, dpi, genotype, sample_group) %>% 

        dplyr::summarise(
            
            value_mean=mean(value), 
            value_sd=sd(value), 
            n=n(), 
            value_se=value_sd / sqrt(n),
            value_se_min=value_mean-value_se, 
            value_se_max=value_mean+value_se, 
            .groups="drop"
        
        ) 

    return(data)
    
}

In [9]:
pl <- function(data, stat) {

    p <- lapply(split(stat, f=stat$measurement), function(x) {

        y_limit <- max(c(abs(x$value_se_min), abs(x$value_se_max)))
    
        ggplot(x, aes(x=dpi, y=value_mean, fill=sample_group, group=genotype)) +
            ggtitle(x$measurement[1]) +
            geom_hline(yintercept=0) +
            geom_bar(
              stat="identity",
              color="black",
              linewidth=0.1,
              width=0.8,
              position=position_dodge(width=0.8)
            ) +
    
    
        geom_point(
          data=data[data$measurement == x$measurement[1], ],
          aes(x=dpi, y=value, fill=sample_group, group=genotype),
          position=position_jitterdodge(
            jitter.width=0.12,
            jitter.height=0,
            dodge.width=0.8,
            seed=1
          ),
          size=1.2,
          alpha=1,
          shape=21,
          colour="black",
          inherit.aes=FALSE
        ) +
    
            geom_errorbar(
              aes(ymin=value_se_min, ymax=value_se_max),
              width=0.4,
              colour="black",
              linewidth=0.25,
              position=position_dodge(width=0.8)
            ) +
        
        scale_fill_manual(values=color$sample_group) +
        facet_grid(~dpi, scales="free") +
        theme(legend.position="none") +
        theme_global_set(4)
        
    }
          )
    
    p <- lapply(p, function(p) egg::set_panel_size(p, width=unit(0.5, "cm"), height=unit(2.0, "cm")))
    p <- do.call(gridExtra::arrangeGrob, c(p, ncol=4, nrow=ceiling(length(p)/4)))
    
    return(p)

}

# Plasma RG374

In [91]:
data <- read.csv("data/RG374/legendplex/plasma.csv") %>% dplyr::mutate(genotype=gsub("'", "", genotype), dpi=ifelse(dpi=="Ctl", "D0", dpi), sample_group=paste(dpi, genotype))

In [92]:
data <- data %>% pivot_longer(cols=-c(tissue, genotype, treatment, dpi, sample_group), names_to="measurement", values_to="value")

In [93]:
data <- iqr(data)
stat <- data_stat(data)
p <- pl(data, stat)

In [94]:
pdf("result/figures/3_validation/legendplex_plasma_RG374.pdf", width=7.5, height=1.9*ceiling(length(p)/5))

grid::grid.draw(p)

dev.off()

pdf 
  2

# Plasma RG401

In [95]:
data <- read.csv("data/RG401/legendplex/plasma.csv") %>% dplyr::mutate(genotype=gsub("'", "", genotype), dpi=ifelse(dpi=="Ctl", "D0", dpi), sample_group=paste(dpi, genotype))

In [96]:
data <- data %>% pivot_longer(cols=-c(tissue, genotype, treatment, dpi, sample_group), names_to="measurement", values_to="value")

In [97]:
data <- iqr(data)
stat <- data_stat(data)
p <- pl(data, stat)

In [98]:
pdf("result/figures/3_validation/legendplex_plasma_RG401.pdf", width=7.5, height=1.9*ceiling(length(p)/5))

grid::grid.draw(p)

dev.off()

pdf 
  2

# Spleen RG374

In [99]:
data <- read.csv("data/RG374/legendplex/spleen.csv") %>% dplyr::mutate(genotype=gsub("'", "", genotype), dpi=ifelse(dpi=="Ctl", "D0", dpi), sample_group=paste(dpi, genotype))

In [100]:
data <- data %>% pivot_longer(cols=-c(tissue, genotype, treatment, dpi, sample_group), names_to="measurement", values_to="value")

In [101]:
data <- iqr(data)
stat <- data_stat(data)
p <- pl(data, stat)

In [102]:
pdf("result/figures/3_validation/legendplex_spleen_RG374.pdf", width=7.5, height=1.9*ceiling(length(p)/5))

grid::grid.draw(p)

dev.off()

pdf 
  2

# Spleen RG401

In [103]:
data <- read.csv("data/RG401/legendplex/spleen.csv") %>% dplyr::mutate(genotype=gsub("'", "", genotype), dpi=ifelse(dpi=="Ctl", "D0", dpi), sample_group=paste(dpi, genotype))

In [104]:
data <- data %>% pivot_longer(cols=-c(tissue, genotype, treatment, dpi, sample_group), names_to="measurement", values_to="value")

In [105]:
data <- iqr(data)
stat <- data_stat(data)
p <- pl(data, stat)

In [106]:
pdf("result/figures/3_validation/legendplex_spleen_RG401.pdf", width=7.5, height=1.9*ceiling(length(p)/5))

grid::grid.draw(p)

dev.off()

pdf 
  2

# Spleen 

In [107]:
data_1 <- read.csv("data/RG374/legendplex/spleen.csv") %>% dplyr::mutate(genotype=gsub("'", "", genotype), dpi=ifelse(dpi=="Ctl", "D0", dpi), sample_group=paste(dpi, genotype))
data_2 <- read.csv("data/RG401/legendplex/spleen.csv") %>% dplyr::mutate(genotype=gsub("'", "", genotype), dpi=ifelse(dpi=="Ctl", "D0", dpi), sample_group=paste(dpi, genotype))

In [108]:
data_1 <- data_1 %>% pivot_longer(cols=-c(tissue, genotype, treatment, dpi, sample_group), names_to="measurement", values_to="value")
data_2 <- data_2 %>% pivot_longer(cols=-c(tissue, genotype, treatment, dpi, sample_group), names_to="measurement", values_to="value")

In [109]:
data <- rbind(data_1, data_2)

In [110]:
data <- iqr(data)
stat <- data_stat(data)
p <- pl(data, stat)

In [111]:
pdf("result/figures/3_validation/legendplex_spleen.pdf", width=7.5, height=1.9*ceiling(length(p)/5))

grid::grid.draw(p)

dev.off()

pdf 
  2